# VVRAG ingestion (free GPU)
Self-contained: run top to bottom, no setup needed. Clones the repo, installs deps, probes the environment (retriever Option A check), ingests, zips `data/index` for download. No logic lives here.

In [ ]:
%cd /content
!if [ -d altVRAG ]; then git -C altVRAG pull -q; else git clone https://github.com/BiprashPandey/altVRAG.git; fi
%cd /content/altVRAG
!pip install -q -e . --no-deps
!mkdir -p data/raw && ls data/raw

Upload your PDFs/slide images into the `altVRAG/data/raw` folder now (Files panel → right-click → Upload), then continue. `data/raw/` exists right after cloning (committed skeleton).

In [ ]:
!nvidia-smi || echo 'NO GPU VISIBLE -> Runtime > Change runtime type > T4 GPU'
%pip install -q torch --index-url https://download.pytorch.org/whl/cu121
%pip install -q colpali-engine qdrant-client pymupdf pillow pytesseract transformers groq openai google-genai
!pip install -q -U torchao  # peft requires torchao for the ColQwen LoRA adapter
!apt-get install -y tesseract-ocr > /dev/null 2>&1
!python -c "import torch; print('torch', torch.__version__, '| cuda build:', torch.version.cuda, '| available:', torch.cuda.is_available())"
!python -c "from vvrag.ingest.embed import probe_environment; probe_environment()"

## 1. Verify the retriever checkpoint (Option A)
Check that `vidore/colqwen2-v1.0` resolves on the Hub (metadata lookup, downloads nothing). If it fails (gated/renamed), log in (`!hf login`) or pick a working ColPali checkpoint and pass `--model-id` in cell 3.

In [ ]:
from huggingface_hub import HfApi
info = HfApi().model_info("vidore/colqwen2-v1.0")  # metadata only, no download
print("id:", info.id, "| private:", info.private, "| gated:", getattr(info, "gated", None))
# 401/GatedRepoError -> run `!hf login` first. 404 -> set MODEL_ID to a verified
# fallback (e.g. vidore/colpali-v1.2) and pass --model-id in the ingest cell.

In [ ]:
!ls data/raw
!python -m vvrag.ingest.run_ingest --input data/raw --out data/index --embedder colqwen --limit 3
# Smoke test ingests 3 files. If manifest looks right, re-run without --limit for the full corpus.

In [ ]:
!cat data/index/manifest.json
!zip -qr index.zip data/index && ls -lh index.zip
# Download index.zip (Files panel) and unzip it into data/index/ on your laptop.
# The zip is self-contained: pages, embeddings, meta.db, manifest AND the qdrant index.